In [9]:
!pip install google-play-scraper

In [10]:
from google_play_scraper import app
import pandas as pd
import numpy as np

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
#scrape jumlah ulasan yang diinginkan
from google_play_scraper import Sort, reviews

result, continuation_token = reviews(
    'id.tix.android',
    lang='id',  #disini kita mau men scrape data ulasan aplikasi tixid yang berada di google play store
    country='id', #kita setting bahasa nya menjadi bahasa indonesia
    sort=Sort.NEWEST, # # kemudian kita gunakan most_relevan untuk mendapatkan ulasan yang paling relevant
    count=10000, # disini jumlah ulasan yang mau kita ambil ada seribu
    filter_score_with=None # # kemudian di filter_rating kita gunakan None untuk mengambil semua rating atau ratting bintang 1 sampai 5
)


In [14]:
data = pd.DataFrame(np.array(result),columns=['review'])
data = data.join(pd.DataFrame(data.pop('review').tolist()))
data.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,f39a90a6-2486-4d9f-9e46-aa054babaa6d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,luar biasa,5,0,4.2.0,2026-01-03 13:58:09,"Hi TIX Hunter, terima kasih atas ulasan yang A...",2026-01-04 03:53:38,4.2.0
1,83226d07-c6df-4444-89b8-86592230d036,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bagus,1,0,4.3.0,2026-01-03 11:13:00,"Hi TIX Hunter, terima kasih atas detail review...",2026-01-04 03:52:24,4.3.0
2,5e7a3c15-0f5f-4833-aca7-8adab7e02f8f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,praktis untuk org yg mager beli tiket buru2,5,0,4.3.0,2026-01-03 07:05:42,"Hi TIX Hunter, terima kasih atas ulasan yang A...",2026-01-03 07:53:40,4.3.0
3,fa2bf213-de38-47b2-9875-de491254ca09,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"aplikasi najis, mau byr pke shopee pay kg bisa...",1,0,None,2026-01-03 04:04:52,"Hi TIX Hunter, mohon maaf atas ketidaknyamanan...",2025-12-21 02:19:33,None
4,5fd17021-3b5e-4b7f-93d6-dbf00def253b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,mantap..mudah,5,0,4.3.0,2026-01-02 16:39:41,"Hi TIX Hunter, terima kasih atas ulasan yang A...",2026-01-03 02:08:45,4.3.0


In [16]:
# Pastikan kolom rating ada dan bernama 'rating'
# Kalau namanya beda, ganti sesuai kolom kamu
print(data.columns)  # cek dulu nama kolom

# Kelompokkan rating jadi dua kategori
def label_sentiment(score):
    if score in [1, 2]:
        return 'Negatif'
    elif score in [3, 4, 5]:
        return 'Positif'

data['kategori'] = data['score'].apply(label_sentiment)

# Hitung jumlah masing-masing kategori
sentiment_count = data['kategori'].value_counts()
sentiment_percent = data['kategori'].value_counts(normalize=True) * 100

# Tampilkan hasil
print("Jumlah masing-masing kategori:")
print(sentiment_count)
print("\nPersentase masing-masing kategori (%):")
print(sentiment_percent)

# (Opsional) buat tabel ringkas
result = pd.DataFrame({
    'Jumlah': sentiment_count,
    'Persentase (%)': sentiment_percent.round(2)
})

print("\nTabel hasil pengelompokan:")
print(result)


Index(['reviewId', 'userName', 'userImage', 'content', 'score',
       'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent',
       'repliedAt', 'appVersion'],
      dtype='object')
Jumlah masing-masing kategori:
kategori
Positif    8234
Negatif    1766
Name: count, dtype: int64

Persentase masing-masing kategori (%):
kategori
Positif    82.34
Negatif    17.66
Name: proportion, dtype: float64

Tabel hasil pengelompokan:
          Jumlah  Persentase (%)
kategori                        
Positif     8234           82.34
Negatif     1766           17.66


In [18]:
data=data[['score', 'content']]#karena kita hanya membutuhkan kolom content dan rating maka kita lakukan filter kolom lgi hingga menyisakan kolom content dan rating.
data.head()

,score,content
0,5,luar biasa
1,1,bagus
2,5,praktis untuk org yg mager beli tiket buru2
3,1,"aplikasi najis, mau byr pke shopee pay kg bisa..."
4,5,mantap..mudah


In [19]:
data.to_csv('/content/drive/MyDrive/data/scrapping_baru.csv', index = False)  #kemudian save menjadi file csv